# Routed decoding

This notebook presents an example of "routed decoding", i.e., how a model can be made to respond differently depending on logical rules on (concept) probes. The general idea of conditioning a response on a property read from activations builds on the CAST algorithm from [Programming Refusal with Conditional Activation Steering](https://arxiv.org/abs/2409.05907), and the execution here reuses the toolkit's phase-plan splicing (the machinery behind `PhasedDecoding`). One of the probes separates advice-seeking from informational questions, which mirrors the use-mention distinction discussed in [When in Doubt, Cascade: Towards Building Efficient and Capable Guardrails](https://ojs.aaai.org/index.php/AIES/article/view/36676).

The driver supports three response strategies: `respond(text)` returns a user-written canned response and generates nothing, `prefix(text)` splices text in front of the model's answer and then generates, and `generate()` passes the row through untouched. This recipe uses `respond` for the referral routes and `generate` for the default.

The router runs one extra forward pass over the prompt (the probe read) to score the probes. This means that a pass-through row costs one prompt forward more than the default decoding path and a canned row costs one prompt forward and zero decode steps.

| component | role |
| --- | --- |
| `StatsSpec`, `ActivationStats` | ambient activation statistics, estimated from a `StatsSpec` and used for whitening |
| `ProbeSet.fit` (with `ProbeFitSpec`, `ContrastivePairs`) | one calibrated linear probe per property, fit on contrastive prompt pools |
| `P`, `Route`, `Router` | boolean predicates over probe names; ordered, first-match-wins routing per row |
| `respond` / `generate` | the two response strategies used here, each lowered to a phase plan |
| `RoutedDecoding` | the decoding driver: one probe read per call, route per row, execute the matched plan |

## Method parameters

The recipe's driver is `RoutedDecoding`, an output-control decoding driver.

| parameter | type | description |
| --- | --- | --- |
| `probes` | `ProbeSet \| ProbeSetFit` | The probes whose decisions drive routing; a `ProbeSetFit` recipe is fit at `steer()` time on the model the pipeline provides |
| `rules` | `Router` | Ordered routes over the probe names; first match wins, evaluated independently per row |
| `allow_model_mismatch` | `bool` | Accept a fit `ProbeSet` whose recorded model fingerprints differ from the pipeline's model |

At generation time the driver also reads an optional `runtime_kwargs` entry, `"canned_responses"` (a per-call override of `respond`/`prefix` text, keyed by route name).

## Setup

If running this from a Google Colab notebook, uncomment and run the following cell to clone and install the toolkit. This is not necessary if running from a local environment where the package has already been installed.

In [1]:
# !git clone https://github.com/IBM/steerability.git
# %cd Steerability
# !pip install -q -e .

In [2]:
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from steerability.algorithms.core.internals import StatsSpec
from steerability.algorithms.core.internals.probes import ProbeFitSpec, ProbeSet, fit_probe
from steerability.algorithms.core.steering_pipeline import SteeringPipeline
from steerability.algorithms.output_control.routed_decoding import (
    P,
    Route,
    RoutedDecoding,
    Router,
    generate,
    respond,
)
from steerability.utils.verbosity import quiet_third_party

quiet_third_party()  # optional: reduce third-party progress bars and info logs

_cwd = Path.cwd()
NOTEBOOK_DIR = _cwd if _cwd.name == "routed_decoding" else _cwd / "examples/notebooks/recipes/routed_decoding"
sys.path.insert(0, str(NOTEBOOK_DIR.resolve()))

from data import (
    EXPECTED_ROUTE,
    FINANCIAL_DEFERRAL,
    HELDOUT_QUERIES,
    LEGAL_DEFERRAL,
    MEDICAL_REFERRAL,
    ambient_texts,
    calibration_data,
    fit_data,
    heldout_rows,
)

We use `ibm-granite/granite-4.1-8b` for this demo. Generation is greedy so the runs are reproducible. A GPU with enough memory for the model is recommended.

In [3]:
MODEL_NAME = "ibm-granite/granite-4.1-8b"

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto", dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"  # batched decoder-only generation; the routed driver strips pads per row either way

gen_params = {
    "max_new_tokens": 80,
    "do_sample": False,
    "pad_token_id": tokenizer.eos_token_id,
}

Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

## The query grid

The probes are fit from small contrastive pools across four domains ({medical, legal, financial, general}) and two asking modes ({info, advice}). The pools, the referral texts, and the held-out set used below live in `data.py` next to this notebook, which also assembles the pools into `ContrastivePairs`. Each domain probe's pairs take positives from both asking modes of the domain and negatives from both modes of every other domain (including general), and the asking-mode probe pairs the advice queries against the informational queries across every domain.

Note that data is constructed in a way to create clear boundaries between domains, e.g., `financial` means the answer requires reasoning about money as a resource (interest, tax, returns, debt, premiums, contributions), while `general` means the decision is about the object or activity itself, with any cost incidental. Examples that span multiple domains (repair-or-replace decisions, extended warranties, lease-versus-buy) belong to both classes and are intentionally excluded. Similarly, `legal` includes consumer-rights situations in everyday vocabulary (delayed flights, refused refunds, gym contracts) and the `general` pools carry the topical near-neighbours with no rights dimension. This helps the probe learn the legal function rather than the courtroom lexicon.

Phrasing is also decorrelated from asking mode. Advice-seeking rotates through many frames ("Should I...", "Is it worth me...", "I can't decide whether...", "What would you do about..."), and informational queries carry first-person context ("My doctor mentioned X -- what does that measure?") and generic-subject "should" ("Why should a wound be kept moist?"). As a result, no single surface cue separates the modes, and the `advice` probe has to read the asking mode itself rather than keying on a template.

In [4]:
for name, pairs in fit_data.items():
    cal = calibration_data[name]
    print(
        f"{name:>9}: fit {len(pairs.positives)} vs {len(pairs.negatives)}, "
        f"calibration {len(cal.positives)} vs {len(cal.negatives)}"
    )

  medical: fit 24 vs 24, calibration 12 vs 12
    legal: fit 24 vs 24, calibration 12 vs 12
financial: fit 24 vs 24, calibration 12 vs 12
   advice: fit 60 vs 60, calibration 36 vs 36


## Fitting the probe set

The `ProbeSet.fit` method fits the probes using the fit pairs (via `data`) and calibrates using the calibration pairs (via `calibration_data`).

The `method="logreg"` argument in `ProbeFitSpec` fits each direction by a regularized logistic regression and `pooling="mean"` aggregates over all prompt tokens.

Note that `"logreg"` (and the default `"lda"`) standardizes features with ambient activation statistics before fitting since the raw residual-stream activations share a large common component and a few outlier coordinates tend to dominate dot products. The standardization is folded into the stored weights allowing for subsequent scoring to be a dot product on raw activations (decision is always `score >= 0`). Additionally note that `ActivationStats` can be saved and reused across every probe fitted on the same model.

In [5]:
stats = StatsSpec(texts=ambient_texts).estimate(model, tokenizer)
print(f"{stats.count} pooled samples over {len(stats.mean)} layers")

spec = ProbeFitSpec(pooling="mean", method="logreg", layer_range=(0.25, 0.75))

probes = ProbeSet.fit(
    model,
    tokenizer,
    data=fit_data,
    spec=spec,
    stats=stats,
    calibration_data=calibration_data,
)

summary_rows = [
    {
        "probe": name,
        "layer": info["layer_ids"][0],
        "method": info["method"],
        "calibrated F1": round(info["f1"], 2),
        "bias": round(info["bias"], 2),
    }
    for name, info in probes.summary().items()
]
pd.DataFrame(summary_rows).set_index("probe")

/dccstor/principled_ai/users/erikmiehling/AISteer360/steerability/algorithms/core/internals/stats.py:59: UserWarning: ActivationStats accumulated 2533 pooled samples, below min_samples=5000. Estimates of per-coordinate variance may be unstable; supply more texts.
  return ActivationStats.estimate(


2533 pooled samples over 40 layers


,layer,method,calibrated F1,bias
probe,,,,
medical,26,logreg,1.0,-0.52
legal,28,logreg,1.0,-0.51
financial,29,logreg,1.0,-1.08
advice,22,logreg,1.0,13.18


## Reading the two axes

The `ProbeSet.read` method scores a batch of prompts against every probe in a single read-only forward pass and returns per-probe signed scores and decisions. The read does not edit any hidden states, so probing leaves generation untouched. Each query is rendered as generation will see it (the user turn plus the generation prompt) before tokenizing, with the chat template supplying its own special tokens.

The four queries below form a two-by-two grid, one topic pair (vaccines and coffee) crossed with the two asking modes. The `medical` column should follow the topic and ignore the mode, and the `advice` column should follow the mode and ignore the topic. Starred entries are fired decisions (`score >= 0`).

Note that the `advice` score on the informational coffee query sits close to zero, so its decision can fall on either side of the threshold. The next section shows how to move the operating point. Under the rules that follow, a marginal `advice` score on its own does not change any behavior since every rule also requires a domain probe to fire.

In [6]:
demo_queries = [
    "How does the immune system respond to a vaccine?",
    "Should I get this vaccine before my trip next month?",
    "How does espresso differ from filter coffee?",
    "Should I switch from filter coffee to espresso in the mornings?",
]

demo_texts = [
    tokenizer.apply_chat_template(
        [{"role": "user", "content": query}], tokenize=False, add_generation_prompt=True
    )
    for query in demo_queries
]
enc = tokenizer(demo_texts, return_tensors="pt", padding=True, add_special_tokens=False)
readout = probes.read(model, enc["input_ids"], enc["attention_mask"])

score_rows = []
for i, query in enumerate(demo_queries):
    row = {"query": query}
    for name in probes.names:
        fired = bool(readout.decisions[name][i])
        row[name] = f"{readout.scores[name][i].item():+.2f}" + (" *" if fired else "")
    score_rows.append(row)
pd.DataFrame(score_rows).set_index("query")

,medical,legal,financial,advice
query,,,,
How does the immune system respond to a vaccine?,+5.23 *,-3.83,-3.82,-9.79
Should I get this vaccine before my trip next month?,+1.76 *,-4.52,-4.63,+8.66 *
How does espresso differ from filter coffee?,-3.39,-3.50,-1.54,-11.64
Should I switch from filter coffee to espresso in the mornings?,-2.53,-5.11,-1.87,+5.02 *


## Moving the operating point

Each probe's threshold is set by the `calibration` argument in `ProbeFitSpec`. Refitting the advice probe with `calibration=("target_fpr", 0.05)` places its operating point at a five percent false-positive rate on the calibration negatives, trading recall for precision. The refit below is for illustration; the routes in the next section keep the `max_f1` calibration fitted above.

In [7]:
strict_spec = ProbeFitSpec(
    pooling="mean", method="logreg", layer_range=(0.25, 0.75), calibration=("target_fpr", 0.05)
)
strict_advice = fit_probe(
    model,
    tokenizer,
    data=fit_data["advice"],
    spec=strict_spec,
    stats=stats,
    calibration_data=calibration_data["advice"],
)

default_advice = probes.probes["advice"]
operating_points = [
    {
        "calibration": "max_f1",
        "bias": round(default_advice.bias, 2),
        "calibration fpr": round(default_advice.meta["calibration"]["fpr"], 2),
    },
    {
        "calibration": "target_fpr = 0.05",
        "bias": round(strict_advice.bias, 2),
        "calibration fpr": round(strict_advice.meta["calibration"]["fpr"], 2),
    },
]
pd.DataFrame(operating_points).set_index("calibration")

,bias,calibration fpr
calibration,,
max_f1,13.18,0.00
target_fpr = 0.05,14.95,0.06


## Routes

A `Router` is defined by an ordered list of routes, each pairing a boolean predicate over probe names with an action. Predicates are built from `P(name)` leaves with `&`, `|`, and `~`. The `route()` method assigns each row its first satisfied route, and rows matching no route fall to the default action, `generate()`, which passes the row to the model untouched. The referral texts (`MEDICAL_REFERRAL`, `LEGAL_DEFERRAL`, `FINANCIAL_DEFERRAL`) are loaded with the query pools and appear in the routed responses below.

Each route here is a conjunction of a domain probe and the asking-mode probe, so a route fires only when both of its probes fire. This means that informational questions on professional topics and everyday advice both take the default, and a marginal score on one axis cannot change behavior on its own.

Note that ordering matters when two domain probes fire on the same query (e.g., a question about the cost of a medical procedure). Since matching stops at the first satisfied route, listing `medical_advice` before `financial_advice` gives it precedence without writing an exclusion (`P("financial") & P("advice") & ~P("medical")`) into the later route.

In [8]:
rules = Router(
    routes=[
        Route("medical_advice", when=P("medical") & P("advice"), action=respond(MEDICAL_REFERRAL)),
        Route("legal_advice", when=P("legal") & P("advice"), action=respond(LEGAL_DEFERRAL)),
        Route("financial_advice", when=P("financial") & P("advice"), action=respond(FINANCIAL_DEFERRAL)),
    ],
    default_action=generate(),
)
print(rules.describe())

Router
├─ 1. medical_advice     if (medical & advice)     -> respond("Questions about your own symptoms, medi…")
├─ 2. legal_advice       if (legal & advice)       -> respond("This is the kind of question I'd rather…")
├─ 3. financial_advice   if (financial & advice)   -> respond("Decisions about your own money -- what …")
└─ default                                         -> generate


## Assembling the pipeline

`RoutedDecoding` pairs the fitted probes (via `probes`) with the rules (via `rules`) and serves as the pipeline's decoding driver. Its `steer()` checks that every probe's recorded model fingerprint matches the pipeline's model and that every probe name the rules reference exists in the set. Note that a `ProbeSetFit` recipe can be passed instead of a fitted set, in which case the driver fits it at steer time on the model the pipeline provides (useful when structural controls produce the final weights inside `steer()`).

In [9]:
routed_decoder = RoutedDecoding(probes=probes, rules=rules)

pipeline = SteeringPipeline(controls=[routed_decoder], model=model, tokenizer=tokenizer)
pipeline.steer()

## A first pass over the stream

We route four queries in one batched call, one for each of the three referral rules and one informational query for the default path. The probe read is a single read-only forward over the batch, a canned row then costs zero decode steps, and a pass-through row generates normally. After the call, `routed_decoder.latest_routes` holds the matched rule name per row (`"default"` for unmatched rows). Each advice query receives its referral in place of the model's own answer and the informational query receives the model's own answer.

In [10]:
routing_demo_queries = [
    "My knee has been swollen for a week -- should I get it looked at?",
    "Should I sign this tenancy agreement if it has no break clause?",
    "Should I overpay my mortgage or put the money into my pension?",
    "What actually happens during a total solar eclipse?",
]
routing_demo_chats = [[{"role": "user", "content": query}] for query in routing_demo_queries]

routed_responses = pipeline.generate(messages=routing_demo_chats, **gen_params)

for query, route, response in zip(routing_demo_queries, routed_decoder.latest_routes, routed_responses):
    print(f"query: {query}")
    print(f"route: {route}")
    print(response)
    print()

query: My knee has been swollen for a week -- should I get it looked at?
route: medical_advice
Questions about your own symptoms, medications, or test results need someone who can examine you and knows your history. Please raise this with your doctor or pharmacist, and seek care promptly if things are getting worse. I'm glad to explain the general medicine behind it if that would help.

query: Should I sign this tenancy agreement if it has no break clause?
route: legal_advice
This is the kind of question I'd rather not answer with generalities, because the right answer depends on your jurisdiction and the specifics of your situation. A licensed attorney can tell you where you actually stand; most local bar associations run referral services with free or low-cost initial consultations, and legal aid organizations can help if cost is a barrier. If deadlines might be involved, such as a notice period or a statute of limitations, it's worth making that call soon.

query: Should I overpay m

## Per-call response overrides

The canned texts live in the rules but can be overridden per call without re-steering. The `"canned_responses"` entry in `runtime_kwargs` maps rule names to replacement text for that call only (keys that do not name a rule carrying canned text are ignored with a warning). Here we replace the medical referral with a shorter weekend message; the route is unchanged and only the text differs.

In [11]:
weekend_referral = (
    "Our advice line is closed for the weekend. For anything urgent, please use "
    "the out-of-hours service; otherwise your own doctor can talk this through "
    "with you next week."
)

response = pipeline.generate(
    messages=routing_demo_chats[0],
    runtime_kwargs={"canned_responses": {"medical_advice": weekend_referral}},
    **gen_params,
)
print(f"route: {routed_decoder.latest_routes[0]}\n\n{response}")

route: medical_advice

Our advice line is closed for the weekend. For anything urgent, please use the out-of-hours service; otherwise your own doctor can talk this through with you next week.


## Held-out routing across the grid

The held-out set covers all eight cells with ten queries each. None of the eighty queries appear in the ninety-six fit or forty-eight calibration queries that produced the probes. The expected route per cell follows from the rules, i.e., advice in one of the three professional domains routes to that domain's referral and every other cell takes the default pass-through.

The professional informational cells test the `advice` probe most directly since each of those queries is one firing `advice` decision away from a referral. The `general` cells check the domain probes on unseen topics (pets, air travel, chess, skiing, pottery) that appear nowhere in the fit or calibration pools, so a domain probe firing on any of them appears as a misroute.

In [12]:
heldout, expected, cell_labels = heldout_rows()
heldout_chats = [[{"role": "user", "content": query}] for query in heldout]
heldout_responses = pipeline.generate(messages=heldout_chats, **gen_params)
heldout_routes = list(routed_decoder.latest_routes)

summary_rows, start = [], 0
for (domain, mode), pool in HELDOUT_QUERIES.items():
    stop = start + len(pool)
    routes = heldout_routes[start:stop]
    exp = EXPECTED_ROUTE[(domain, mode)]
    observed = ", ".join(
        f"{route} x{count}" if count > 1 else route for route, count in Counter(routes).items()
    )
    summary_rows.append(
        {
            "cell": f"{domain} / {mode}",
            "expected route": exp,
            "correct": f"{sum(route == exp for route in routes)}/{len(pool)}",
            "observed routes": observed,
        }
    )
    start = stop

pd.DataFrame(summary_rows).set_index("cell")

,expected route,correct,observed routes
cell,,,
medical / info,default,10/10,default x10
medical / advice,medical_advice,10/10,medical_advice x10
legal / info,default,10/10,default x10
legal / advice,legal_advice,10/10,legal_advice x10
financial / info,default,10/10,default x10
financial / advice,financial_advice,10/10,financial_advice x10
general / info,default,10/10,default x10
general / advice,default,10/10,default x10


In [13]:
n_correct = sum(got == exp for got, exp in zip(heldout_routes, expected))
print(f"overall routing accuracy: {n_correct}/{len(heldout)}")

scores = routed_decoder.probes.latest.scores
misses = [i for i, (got, exp) in enumerate(zip(heldout_routes, expected)) if got != exp]
for i in misses:
    detail = ", ".join(f"{name} {scores[name][i].item():+.2f}" for name in probes.names)
    print(f"\nmisrouted ({cell_labels[i]} -> {heldout_routes[i]}): {heldout[i]}\n  probe scores: {detail}")
if not misses:
    print("\nno misrouted queries in this run")

overall routing accuracy: 80/80

no misrouted queries in this run


## Summary

This recipe reads two properties of each query from the model's hidden states and uses their combination to select a response strategy. Four probes cover the eight-cell grid, with three domain probes and one asking-mode probe. Each probe is fitted on a small contrastive pool that varies only along its target axis, calibrated on a disjoint set, and validated against the model by fingerprint. The pools preserve a consistent label boundary by excluding straddlers and include phrasings from both asking modes so that the probes detect the target properties rather than a phrasing template.

Each rule combines one domain probe with the asking-mode probe, so the policy acts only when both conditions are satisfied. If two domain probes fire for the same query, rule order determines the route because matching stops at the first satisfied rule. The canned referral is spliced in one prompt-forward step with no decoding, the selected route is reported through `latest_routes`, and each probe's operating point is a calibration parameter.

The [routing versus prompting study](../../studies/routing_vs_prompting.ipynb) compares this recipe against two prompting baselines on the held-out grid, measuring routing accuracy, fidelity to the response texts, token cost, disturbance of the default path, and robustness to a counter-instruction. Since the routed pipeline is an ordinary steering pipeline, it can also be run over a task set and scored with the evaluation stack (`SteeringEval`).